In [37]:
import json

with open("../data/raw/videos_raw.json", "r", encoding="utf-8") as f:
    videos = json.load(f)

print(f"✅ {len(videos)} vidéos chargées")

✅ 968 vidéos chargées


In [ ]:
import json
import re

with open("../data/raw/videos_raw.json", "r", encoding="utf-8") as f:
    videos = json.load(f)

def convertir_duree(iso):
    if not iso:
        return 0
    match = re.match(r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?', iso)
    if not match:
        return 0
    heures   = int(match.group(1) or 0)
    minutes  = int(match.group(2) or 0)
    secondes = int(match.group(3) or 0)
    return heures * 3600 + minutes * 60 + secondes

# ✅ Classification par durée (demandée par le prof)
def classifier_video(duree_sec):
    if duree_sec < 60:
        return "short"       # moins de 60 sec
    elif duree_sec < 3600:
        return "video"       # entre 1 min et 1h
    elif duree_sec < 7200:
        return "stream"      # entre 1h et 2h
    else:
        return "podcast"     # plus de 2h

dataset = []
for video in videos:
    duree = convertir_duree(video["duration"])
    titre = (video["title"] or "").strip()
    ligne = {
        "videoId"      : video["videoId"],
        "title"        : titre,
        "date"         : (video["publishedAt"] or "")[:10],
        "duree_sec"    : duree,
        "vues"         : int(video["viewCount"]    or 0),
        "likes"        : int(video["likeCount"]    or 0),
        "commentaires" : int(video["commentCount"] or 0),
        "type"         : classifier_video(duree),   # ✅ 
    }
    dataset.append(ligne)

# ✅ Vérification de la distribution
from collections import Counter
types = Counter(v["type"] for v in dataset)

print(f"✅ {len(dataset)} vidéos prêtes")
print(f"Clés : {list(dataset[0].keys())}")
print(f"─────────────────────────────────")
print(f"📱 Shorts   : {types['short']}")
print(f"🎥 Videos   : {types['video']}")
print(f"📺 Streams  : {types['stream']}")
print(f"🎙️ Podcasts : {types['podcast']}")
print(f"─────────────────────────────────")
print(f"Total       : {sum(types.values())}")

✅ 968 vidéos prêtes
Clés : ['videoId', 'title', 'date', 'duree_sec', 'vues', 'likes', 'commentaires', 'type']
─────────────────────────────────
📱 Shorts   : 189
🎥 Videos   : 768
📺 Streams  : 2
🎙️ Podcasts : 9
─────────────────────────────────
Total       : 968


In [43]:
print(f"Dataset propre : {len(dataset)} vidéos")

print("Exemple vidéo 1 :")
for cle, valeur in dataset[0].items():
    print(f"  {cle} : {valeur}")

Dataset propre : 968 vidéos
Exemple vidéo 1 :
  videoId : pzxtGL9kzPs
  title : Try Not To Get Slimed
  date : 2026-04-24
  vues : 41032003
  likes : 540885
  commentaires : 8492


In [2]:

# SÉPARATION DES VIDÉOS EN CATÉGORIES PAR DURÉE

# 4 listes vides pour stocker chaque catégorie
shorts   = []  # Courtes vidéos  : moins de 60 secondes
vids     = []  # Vidéos normales : entre 60 sec et 1h (3600 sec)
streams  = []  # Streams/Lives   : entre 1h et 2h (7200 sec)
podcasts = []  # Podcasts        : plus de 2h (7200 sec)

# On parcourt les 968 vidéos une par une
for video in dataset:
    
    # On récupère la durée en secondes de chaque vidéo
    duree = video["duree_sec"]

    # Moins de 60 sec → Short
    if duree < 60:
        shorts.append(video)

    # Entre 60 sec et 3600 sec (1h) → Vidéo normale
    elif duree < 3600:
        vids.append(video)

    # Entre 3600 sec (1h) et 7200 sec (2h) → Stream/Live
    elif duree < 7200:
        streams.append(video)

    # Plus de 7200 sec (2h) → Podcast
    else:
        podcasts.append(video)


# VÉRIFICATION — afficher le nombre de vidéos par catégorie

print(f"📱 Shorts   : {len(shorts)}")
print(f"🎥 Vids     : {len(vids)}")
print(f"📺 Streams  : {len(streams)}")
print(f"🎙️ Podcasts : {len(podcasts)}")
print(f"─────────────────────────────")
# Total doit être égal à 968
print(f"Total       : {len(shorts) + len(vids) + len(streams) + len(podcasts)}")

📱 Shorts   : 189
🎥 Vids     : 768
📺 Streams  : 2
🎙️ Podcasts : 9
─────────────────────────────
Total       : 968


In [3]:
print(dataset[0].keys())

dict_keys(['videoId', 'title', 'date', 'duree_sec', 'vues', 'likes', 'commentaires', 'type'])
